<a href="https://colab.research.google.com/github/kangwonlee/nmisp/blob/dependabot/pip/tests/lxml-6.1.0/15_optimization/035_colab_mnist_keras_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


## 2.1 A first look at a neural network

* from : F. Chollet, Deep Learning with Python, ISBN 9781617294433, 2017
* https://github.com/fchollet/deep-learning-with-python-notebooks
* https://www.manning.com/books/deep-learning-with-python



> _Explanation added by Claude (Anthropic) — beginner-friendly notes._
> _이하 설명은 Claude (Anthropic) 가 초보자용으로 추가한 것입니다._

**What this notebook does · 이 노트북이 하는 일**

**MNIST** is a dataset of 70,000 handwritten digit images ($28 \times 28$ pixels, grayscale, labels $0$–$9$). It's the "hello world" of deep learning — simple enough to train quickly, complex enough that you need a real model.
**MNIST** 는 손글씨 숫자 이미지 7만 장을 모은 데이터셋이다 ($28 \times 28$ 픽셀, 흑백, 라벨은 $0$–$9$). 딥러닝의 "hello world" — 빠르게 학습되면서도, 제대로 된 모델이 필요한 수준.

We'll build a small neural network in **Keras** (TensorFlow's high-level API) and train it to classify digits. Compare this with notebook 030, which used only 2 weights on a simple 2D problem — here we'll have hundreds of thousands of weights, but the principle is the same: **minimize a cost function by adjusting weights.**
**Keras** (TensorFlow 의 고수준 API) 로 작은 신경망을 만들어 숫자를 분류한다. 노트북 030 에서는 2차원 문제에 가중치 2개만 썼다. 여기서는 수십만 개의 가중치를 쓰지만 원리는 같다: **비용함수를 최소화하도록 가중치를 조정한다.**


In [ ]:
import os
import pathlib

import numpy as np
import pandas as pd
import tensorflow as tf



### Listing 2.1 Loading the MNIST dataset in Keras



In [ ]:
try:
  data_folder = pathlib.Path('sample_data')
  assert data_folder.exists()
  assert data_folder.is_dir()

  def read_data(data_path):
    df = pd.read_csv(
        data_path,
        header=None
    )
    labels = np.array(df.iloc[:, 0])
    images = np.array(df.iloc[:, 1:])

    return images, labels


  train_images, train_labels = read_data(data_folder / 'mnist_train_small.csv')
  test_images, test_labels = read_data(data_folder / 'mnist_test.csv')

except AssertionError:
  (train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()



### Listing 2.2 The training data


In [ ]:
train_images.shape



In [ ]:
n_train = len(train_labels)



In [ ]:
train_labels



### Listing 2.3 The test data


In [ ]:
test_images.shape


In [ ]:
n_test = len(test_labels)



In [ ]:
test_labels


### Listing 2.4 The network architecture



> _Explanation added by Claude (Anthropic) — beginner-friendly notes._
> _이하 설명은 Claude (Anthropic) 가 초보자용으로 추가한 것입니다._

**Understanding the architecture · 모델 구조 이해**

The network below is a stack of **2 dense (fully-connected) layers**:
아래 신경망은 **완전연결(dense) 층 2개**를 쌓은 구조다:

- **Input**: a flattened $28 \times 28 = 784$-dimensional vector (one pixel = one number in $[0, 1]$).
  입력: $28 \times 28 = 784$ 차원 벡터 (픽셀 하나가 $[0, 1]$ 범위의 숫자 하나).
- **Hidden layer (512 units, ReLU)**: computes $\text{ReLU}(W_1 x + b_1)$ where $W_1$ is $(512, 784)$. ReLU is $\max(0, z)$ — a modern replacement for the sigmoid from notebook 030.
  은닉층 (512개, ReLU): $\text{ReLU}(W_1 x + b_1)$. $W_1$ 의 크기는 $(512, 784)$. ReLU 는 $\max(0, z)$ — 노트북 030 의 시그모이드를 대체하는 현대적 활성함수.
- **Output layer (10 units, softmax)**: produces 10 probabilities that sum to $1$ — one per digit class.
  출력층 (10개, softmax): 합이 $1$ 인 확률 10개를 낸다 — 숫자 $0$–$9$ 각각의 확률.

Total trainable weights: $784 \times 512 + 512 + 512 \times 10 + 10 = 407{,}050$. The optimizer's job is to find good values for all of them.
전체 학습 가중치 수: $784 \times 512 + 512 + 512 \times 10 + 10 = 407{,}050$ 개. 최적화기는 이 모두의 좋은 값을 찾아야 한다.


In [ ]:
import keras

network = keras.models.Sequential()
network.add(keras.layers.Input(shape=(28 * 28,)))  # Define input shape using Input layer
network.add(keras.layers.Dense(512, activation='relu'))
network.add(keras.layers.Dense(10, activation='softmax'))



### 2.5 The compilation step



> _Explanation added by Claude (Anthropic) — beginner-friendly notes._
> _이하 설명은 Claude (Anthropic) 가 초보자용으로 추가한 것입니다._

**The three ingredients of `.compile()` · `.compile()` 의 세 가지 재료**

Before training, Keras needs to know:
학습 전에 Keras는 다음 세 가지를 알아야 한다:

| Argument | Role · 역할 | Here · 여기서는 |
|----------|-------------|-----------------|
| `optimizer` | Algorithm that adjusts weights. 가중치를 조정하는 알고리즘. | `rmsprop` — an adaptive-learning-rate variant of SGD. |
| `loss` | What we minimize (like cost function in 030). 최소화할 대상 (030 의 비용함수와 같은 개념). | `categorical_crossentropy` — the multi-class version of cross-entropy from 030. |
| `metrics` | What we report but don't directly optimize. 보고용 지표 (직접 최적화하지는 않음). | `accuracy` — fraction of correct predictions. |

Note: loss is used *by the optimizer*; accuracy is just for humans to read.
비용(loss)은 최적화기가 쓰고, 정확도(accuracy)는 사람이 보기 위한 것.


In [ ]:
network.compile(
    optimizer='rmsprop',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)



### 2.6 Preparing the image data



> _Explanation added by Claude (Anthropic) — beginner-friendly notes._
> _이하 설명은 Claude (Anthropic) 가 초보자용으로 추가한 것입니다._

**Why reshape and divide by 255 · 왜 reshape 하고 255 로 나누는가?**

- **Reshape** $(n, 28, 28) \to (n, 784)$: the dense layer expects a flat vector per sample, not a 2D image. We throw away spatial structure here — CNNs (next step up) preserve it.
  **reshape** $(n, 28, 28) \to (n, 784)$: dense 층은 샘플당 1D 벡터를 기대한다. 여기서는 공간 구조를 버린다. CNN (다음 단계) 은 공간 구조를 유지한다.
- **Divide by 255**: pixel values originally range $[0, 255]$. Neural networks train much better when inputs are in a small range around $0$ (here $[0, 1]$). Large input magnitudes can make gradients explode or weights grow unstable.
  **255 로 나누기**: 픽셀 값은 원래 $[0, 255]$ 범위. 신경망은 입력이 $0$ 근처의 작은 범위에 있을 때 훨씬 잘 학습한다 (여기서는 $[0, 1]$). 입력이 너무 크면 기울기 폭발·가중치 불안정이 쉽게 일어난다.


In [ ]:
train_images = np.array(train_images).reshape((n_train, 28 * 28))
train_images = np.array(train_images).astype('float32') / 255

test_images = np.array(test_images).reshape((n_test, 28 * 28))
test_images = np.array(test_images).astype('float32') / 255



### 2.7 Preparing the labels



> _Explanation added by Claude (Anthropic) — beginner-friendly notes._
> _이하 설명은 Claude (Anthropic) 가 초보자용으로 추가한 것입니다._

**One-hot encoding · 원-핫 인코딩**

Labels are originally integers $0$–$9$. We convert each label $k$ into a 10-dimensional vector of zeros with a single $1$ at position $k$:
라벨은 원래 정수 $0$–$9$. 각 라벨 $k$ 를 길이 10짜리 벡터로 바꾼다. 위치 $k$ 만 $1$, 나머지는 $0$:

$$
3 \mapsto [0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
$$

Why? Because the network's output is a length-10 probability vector (from softmax). We need the target in the same shape to compute cross-entropy.
왜? 신경망 출력이 길이 10 의 확률 벡터(softmax)이기 때문. 교차 엔트로피를 계산하려면 정답도 같은 모양이어야 한다.

An alternative is `sparse_categorical_crossentropy`, which accepts integer labels directly — same math, just a different API.
`sparse_categorical_crossentropy` 를 쓰면 정수 라벨을 바로 받을 수도 있다 — 수식은 같고 API만 다르다.


In [ ]:
train_labels = keras.utils.to_categorical(train_labels)
test_labels = keras.utils.to_categorical(test_labels)



### 2.8 Training the network



> _Explanation added by Claude (Anthropic) — beginner-friendly notes._
> _이하 설명은 Claude (Anthropic) 가 초보자용으로 추가한 것입니다._

**Epoch vs batch size · 에폭과 배치크기**

- **Epoch** = one full pass over the training data. `n_epoch = 5` means the optimizer sees every training image 5 times.
  **Epoch** = 훈련 데이터 전체를 한 번 훑는 것. `n_epoch = 5` 면 각 이미지를 5회 본다.
- **Batch size** (128 below) = how many images are processed together to compute one gradient step. Too small: noisy updates. Too large: slow and may underfit. 128 is a common default.
  **Batch size** (아래 128) = 한 번의 기울기 계산에 쓰이는 이미지 수. 너무 작으면 노이즈가 많고, 너무 크면 느리고 덜 학습된다. 128 이 흔한 기본값.

With 60,000 training images and batch size 128, one epoch is $\lceil 60000 / 128 \rceil = 469$ gradient updates. Five epochs = $\sim 2{,}345$ updates total.
훈련 이미지 6만, 배치 128 이면 한 에폭에 $\lceil 60000 / 128 \rceil = 469$ 번의 기울기 갱신. 5 에폭이면 총 $\sim 2{,}345$ 번.

The `%%time` magic at the start of the cell prints how long the cell took to run — useful for comparing CPU (this notebook) with GPU (036).
셀 첫 줄의 `%%time` 매직은 셀 실행 시간을 출력한다 — CPU (이 노트북) 와 GPU (036) 를 비교하기 좋다.


In [ ]:
n_epoch = 1 if os.getenv('GITHUB_ACTIONS', False) else 5



In [ ]:
%%time
network.fit(train_images, train_labels, epochs=n_epoch, batch_size=128)



### 2.9 Evaluating the network



> _Explanation added by Claude (Anthropic) — beginner-friendly notes._
> _이하 설명은 Claude (Anthropic) 가 초보자용으로 추가한 것입니다._

**Training vs test accuracy · 훈련 정확도와 시험 정확도**

`network.fit` reported ~99% accuracy on training data. `network.evaluate` below reports accuracy on **test** data — images the model has **never seen**. The gap between training and test accuracy tells us about **overfitting**: if training is 99% but test is 85%, the model memorized rather than generalized.
`network.fit` 은 훈련 데이터에 대해 99% 정도의 정확도를 보였을 것이다. 아래 `network.evaluate` 는 **시험** 데이터(모델이 **한 번도 보지 못한** 이미지)에 대한 정확도다. 이 둘의 차이는 **과적합(overfitting)** 을 나타낸다: 훈련 99%, 시험 85% 라면 모델이 일반화보다는 암기를 한 것.

For MNIST with this small model, the gap is usually tiny because MNIST is easy. Real-world datasets are much harder, and managing the train–test gap becomes a core skill.
MNIST 는 쉬운 과제이고 모델도 작아서 보통 차이가 거의 없다. 실제 데이터셋은 훨씬 어렵고, 훈련-시험 격차 관리는 핵심 역량이 된다.


In [ ]:
test_loss, test_acc = network.evaluate(test_images, test_labels)
print('test_acc:', test_acc)

